# Parameterized Logical CNOT via Homological Measurement

This notebook builds the same logical CNOT experiment for any compatible input code. The builder supports either one shared code that contains both control and target logical qubits, or separate control and target codes, while the ancilla always has its own code.

In [1]:
import numpy as np
import stim

from epic import (
    AllocCode,
    FreeCode,
    InitCode,
    PauliChar,
    PauliEigenState,
    PauliString,
    QECCompiler,
    ReadoutCode,
    RotatedSurfaceCode,
    StimLikeNoiseModel,
 )
from epic.modules.qec_gadgets.pauli_product_measurement.homological_measurement import HomologicalMeasurement
from epic.modules.stabilizers_codes.css_code import CSSCode


def _logical_qubit_varnames(code, labeled_logicals=None, auxiliary_prefix=None):
    labeled_logicals = labeled_logicals or {}
    if labeled_logicals:
        max_index = max(labeled_logicals)
        if max_index >= code.k:
            raise IndexError("A labeled logical index is out of range for the code.")
    auxiliary_prefix = auxiliary_prefix or f"{code.name}_aux"
    logical_names = [f"{auxiliary_prefix}_{i}" for i in range(code.k)]
    for logical_index, label in labeled_logicals.items():
        logical_names[logical_index] = label
    return logical_names


def _readout_tag(code_varname, code, logical_index):
    return f"readout_LZ_{code_varname}_{code.logical_qubits[logical_index].name}"


def build_cnot_experiment(
    compiler_config,
    control_code,
    ancilla_code,
    target_code=None,
    noise=1e-3,
):
    compiler = QECCompiler(config=compiler_config)
    control_code_varname="control_patch"
    target_code_varname="target_patch"
    shared_code_varname="data_patch"
    ancilla_code_varname="ancilla_patch"
    control_label="control"
    target_label="target"
    ancilla_label="ancilla"

    shared_data_code = target_code is None or target_code is control_code
    if shared_data_code:
        if control_code.k < 2:
            raise ValueError("The shared control code must have at least two logical qubits.")
        data_specs = [
            (
                control_code,
                shared_code_varname,
                {0: control_label, 1: target_label},
            )
        ]
        control_observable = _readout_tag(shared_code_varname, control_code, 0)
        target_observable = _readout_tag(shared_code_varname, control_code, 1)
    else:
        if control_code.k < 1:
            raise ValueError("The control code must have at least one logical qubit.")
        if target_code.k < 1:
            raise ValueError("The target code must have at least one logical qubit.")
        data_specs = [
            (
                control_code,
                control_code_varname,
                {0: control_label},
            ),
            (
                target_code,
                target_code_varname,
                {0: target_label},
            ),
        ]
        control_observable = _readout_tag(control_code_varname, control_code, 0)
        target_observable = _readout_tag(target_code_varname, target_code, 0)

    if not 0 <= ancilla_code.k:
        raise IndexError("ancilla_code must have at least one logical qubit.")
    ancilla_logical_varnames = _logical_qubit_varnames(
        ancilla_code,
        {0: ancilla_label},
        auxiliary_prefix=f"{ancilla_code.name}_aux",
    )

    program = []
    for code, code_varname, labeled_logicals in data_specs:
        program.append(
            AllocCode(
                target_code=code,
                code_varname=code_varname,
                logical_qubits_varnames=_logical_qubit_varnames(
                    code,
                    labeled_logicals,
                    auxiliary_prefix=f"{code.name}_aux",
                ),
            )
        )
    program.append(
        AllocCode(
            target_code=ancilla_code,
            code_varname=ancilla_code_varname,
            logical_qubits_varnames=ancilla_logical_varnames,
        )
    )

    for _, code_varname, _ in data_specs:
        program.append(
            InitCode(
                targets=[code_varname],
                initial_state=PauliEigenState.Z_plus,
                tag=f"init_{code_varname}",
            )
        )
    program.append(
        InitCode(
            targets=[ancilla_code_varname],
            initial_state=PauliEigenState.X_plus,
            tag=f"init_{ancilla_code_varname}",
        )
    )

    program.append(
        HomologicalMeasurement(
            targets=[control_label, ancilla_label],
            product_to_measure=PauliString(string=(PauliChar.Z, PauliChar.Z)),
            tag="MZZ_ca",
        )
    )
    program.append(
        HomologicalMeasurement(
            targets=[ancilla_label, target_label],
            product_to_measure=PauliString(string=(PauliChar.X, PauliChar.X)),
            tag="MXX_at",
        )
    )

    for _, code_varname, _ in data_specs:
        program.append(ReadoutCode(targets=[code_varname], tag=f"LZ_{code_varname}"))
    program.append(
        ReadoutCode(targets=[ancilla_code_varname], tag=f"LZ_{ancilla_code_varname}")
    )

    for code_varname in [spec[1] for spec in data_specs] + [ancilla_code_varname]:
        program.append(FreeCode(code_varname=code_varname))

    compiled_program = compiler.compile(program)

    noise_model = StimLikeNoiseModel.from_stim_like_probabilities(
        after_clifford_depolarization=noise,
        after_reset_flip_probability=noise,
        before_measure_flip_probability=noise,
    )

    stim_observables = [
        [control_observable],
        [
            "hm_PP_MZZ_ca",
            _readout_tag(ancilla_code_varname, ancilla_code, 0),
            target_observable,
        ],
    ]

    stim_program = compiled_program.to_stim_program(
        stim_observables,
        noise_model=noise_model,
    )
    circuit = stim.Circuit(stim_program)

    return circuit

## Compiler Configuration

As in the other homological-measurement examples, we use ZX-coloring extraction for syndrome primitives and set objective distance to 3.

In [2]:
compiler_config = {
    "objective_distance": 3,
    "primitives": {
        "epic.core.qec_primitives.interfaces.ApplyGate": "epic.modules.qec_primitives.apply_gates.simple_gate_application.SimpleGateApplication",
        "epic.core.qec_primitives.interfaces.Readout": "epic.modules.qec_primitives.readouts.naive_readout.NaiveReadout",
        "epic.core.qec_primitives.interfaces.ExtractSyndrome": "epic.modules.qec_primitives.syndrome_extraction.zxcoloring_extraction.ZXColoringExtraction",
    },
}



## Compile and Build Stim Circuit

We compile the program, inspect emitted observables, then define deterministic Stim observables corresponding to the logical CNOT action.

Feedforward convention (same idea as in getting-started):
- output control observable: direct Z readout of HGP logical qubit 0 (`control`)
- output target observable: $m(ZZ_{ca}) \oplus m(Z_a) \oplus m(Z_{\mathrm{target}})$, where target is HGP logical qubit 1

In [3]:
import qec

from qec.code_constructions import HypergraphProductCode
import numpy as np

hamming_code = np.array([[1, 0, 0, 1, 0, 1, 1],
                         [0, 1, 0, 1, 1, 0, 1],
                         [0, 0, 1, 0, 1, 1, 1]])

hgp = HypergraphProductCode(hamming_code, hamming_code)
hx = hgp.x_stabilizer_matrix.toarray().astype(np.uint8)
hz = hgp.z_stabilizer_matrix.toarray().astype(np.uint8)

x_logicals, z_logicals = hgp.compute_logical_basis()
logical_qubits = [
    (
        x_logicals.getrow(i).toarray().ravel().astype(int).tolist(),
        z_logicals.getrow(i).toarray().ravel().astype(int).tolist(),
    )
    for i in range(x_logicals.shape[0])
 ]

hgp_code = CSSCode.from_css_pcm(
    code_name="hgp",
    hx=hx,
    hz=hz,
    logical_qubits=logical_qubits,
)

ancilla_code = RotatedSurfaceCode.from_distance(
    distance=3,
    code_name="ancilla_d3",
    system_coordinate=(1, 0),
)

noise = 1e-3
circuit = build_cnot_experiment(
    compiler_config,
    control_code=hgp_code,
    ancilla_code=ancilla_code,
    target_code=None,
    noise=noise,
)

len(circuit.shortest_graphlike_error(ignore_ungraphlike_errors=True))


2

In [ ]:
from typing import List

import numpy as np
import sinter
from ldpc.sinter_decoders import SinterBpOsdDecoder
from matplotlib import pyplot as plt

plt.set_loglevel("warning")


def generate_tasks():
    noises = np.geomspace(1e-6, 1e-2, 5)
    for d in [3]:
        for noise in noises:
            yield sinter.Task(
                circuit=build_cnot_experiment(
                    compiler_config,
                    control_code=hgp_code,
                    ancilla_code=ancilla_code,
                    target_code=None,
                    noise=noise,
                ),
                json_metadata={
                    "d": d,
                    "p": float(noise),
                    "rounds": 9,
                },
            )


def run_experiment() -> List[sinter.TaskStats]:
    return sinter.collect(
        num_workers=4,
        max_shots=100,
        max_errors=100,
        tasks=generate_tasks(),
        decoders=["bposd"],
        custom_decoders={
            "bposd": SinterBpOsdDecoder(
                max_iter=5,
                bp_method="ms",
                ms_scaling_factor=0.625,
                schedule="parallel",
                osd_method="OSD_0",
            ),
        },
        print_progress=True,
    )


def print_results(stats: List[sinter.TaskStats]) -> None:
    print(sinter.CSV_HEADER)
    for stat in stats:
        print(stat.to_csv_line())


def plot_results(stats: List[sinter.TaskStats]):
    fig, ax = plt.subplots(1, 1, figsize=(7, 4.5))
    sinter.plot_error_rate(
        ax=ax,
        stats=stats,
        group_func=lambda stat: f"{hgp_code.name} d={stat.json_metadata['d']}",
        filter_func=lambda stat: stat.decoder == "bposd",
        x_func=lambda stat: stat.json_metadata["p"],
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title("HGP Memory Experiment with BP+OSD")
    ax.set_xlabel("Physical Error Rate")
    ax.set_ylabel("Logical Error Rate per Shot")
    ax.grid(which="major")
    ax.grid(which="minor")
    ax.legend()
    return fig, ax


collected_stats = run_experiment()
print_results(collected_stats)
fig, ax = plot_results(collected_stats)
fig.set_dpi(120)

Starting 4 workers...
5 tasks left:
  workers decoder eta shots_left errors_left json_metadata        
        1   bposd   ?       1000         100 d=3,p=1e-06,rounds=9 
        1   bposd   ?       1000         100 d=3,p=1e-05,rounds=9 
        1   bposd 14m        999         100 d=3,p=0.0001,rounds=9
        1   bposd   ?       1000         100 d=3,p=0.001,rounds=9 
        0   bposd ?·∞       1000         100 d=3,p=0.01,rounds=9  
5 tasks left:
  workers decoder eta shots_left errors_left json_metadata        
        1   bposd 14m        999         100 d=3,p=1e-06,rounds=9 
        1   bposd 15m        999         100 d=3,p=1e-05,rounds=9 
        1   bposd 14m        999         100 d=3,p=0.0001,rounds=9
        1   bposd 15m        999         100 d=3,p=0.001,rounds=9 
        0   bposd ?·∞       1000         100 d=3,p=0.01,rounds=9  
5 tasks left:
  workers decoder eta shots_left errors_left json_metadata        
        1   bposd 13m        998         100 d=3,p=1e-06,rounds=9

KeyboardInterrupt: 